# Credit Card Fraud Detection — Full EDA
**Purpose:** Validate every assumption baked into the MLOps pipeline code.

Upload `creditcard.csv` when Colab prompts, then run all cells top to bottom.

In [ ]:
# ── Install dependencies
!pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost shap

In [ ]:
from google.colab import files
print('Upload creditcard.csv now...')
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('creditcard.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 1. Schema & Basic Stats

In [ ]:
print('=== DTYPES ===')
print(df.dtypes)
print()
print('=== NULL COUNTS ===')
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else 'No nulls — clean dataset')
print()
print('=== DUPLICATE ROWS ===')
n_dupes = df.duplicated().sum()
print(f'{n_dupes:,} duplicates ({n_dupes/len(df):.4%} of rows)')

In [ ]:
print('=== CLASS DISTRIBUTION (assumption: ~0.17% fraud) ===')
vc = df['Class'].value_counts()
print(vc)
fraud_rate = df['Class'].mean()
print(f'\nFraud rate: {fraud_rate:.6%}')
print(f'scale_pos_weight would be: {(df["Class"]==0).sum() / (df["Class"]==1).sum():.1f}')
print(f'\nPipeline assumption was: ~0.17% fraud, scale_pos_weight ~578')
print(f'ACTUAL:                  {fraud_rate:.4%} fraud, scale_pos_weight {(df["Class"]==0).sum()/(df["Class"]==1).sum():.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
vc.plot(kind='bar', ax=axes[0], color=['steelblue','crimson'], edgecolor='white')
axes[0].set_title('Class Distribution (raw counts)')
axes[0].set_xticklabels(['Legitimate (0)', 'Fraud (1)'], rotation=0)
axes[0].set_ylabel('Count')
for i, v in enumerate(vc):
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(df):.3%})', ha='center', fontsize=9)

# Pie
axes[1].pie(vc, labels=['Legitimate', 'Fraud'], autopct='%1.3f%%',
            colors=['steelblue', 'crimson'], startangle=90)
axes[1].set_title('Class Balance')
plt.suptitle('Class Imbalance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Time Feature Analysis
Pipeline assumption: drop `Time` — validate this.

In [ ]:
print('=== TIME FEATURE ===')
print(f'Range: [{df["Time"].min():.0f}, {df["Time"].max():.0f}] seconds')
print(f'That is {df["Time"].max()/3600:.1f} hours of data (~{df["Time"].max()/86400:.1f} days)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Time distribution by class
df[df['Class']==0]['Time'].hist(bins=60, ax=axes[0], alpha=0.6, label='Legitimate', color='steelblue', density=True)
df[df['Class']==1]['Time'].hist(bins=60, ax=axes[0], alpha=0.8, label='Fraud', color='crimson', density=True)
axes[0].set_title('Time Distribution by Class (density)')
axes[0].set_xlabel('Time (seconds)')
axes[0].legend()

# Fraud rate over time bins
df['time_hour'] = (df['Time'] // 3600).astype(int)
fraud_by_hour = df.groupby('time_hour')['Class'].mean()
fraud_by_hour.plot(ax=axes[1], marker='o', color='crimson', linewidth=1.5)
axes[1].set_title('Fraud Rate by Hour')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Fraud rate')
axes[1].axhline(fraud_rate, color='gray', linestyle='--', label=f'Overall mean ({fraud_rate:.4%})')
axes[1].legend()

plt.suptitle('Time Feature — Should We Drop It?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation of Time with Class
corr = df['Time'].corr(df['Class'])
print(f'\nCorrelation of Time with Class: {corr:.6f}')
print('Verdict: if fraud rate varies meaningfully by hour → Time has signal, consider keeping')

## 3. Amount Feature Analysis
Pipeline assumption: RobustScaler is appropriate — validate this.

In [ ]:
print('=== AMOUNT STATS ===')
print(df.groupby('Class')['Amount'].describe().round(2))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Raw distribution
df[df['Class']==0]['Amount'].clip(upper=2000).hist(bins=80, ax=axes[0,0], alpha=0.6, color='steelblue', density=True)
df[df['Class']==1]['Amount'].clip(upper=2000).hist(bins=80, ax=axes[0,0], alpha=0.8, color='crimson', density=True)
axes[0,0].set_title('Amount Distribution (clipped at $2000)')
axes[0,0].set_xlabel('Amount ($)')
axes[0,0].legend(['Legitimate', 'Fraud'])

# Log scale
log_amount = np.log1p(df['Amount'])
df_plot = df.copy()
df_plot['log_amount'] = log_amount
df_plot[df_plot['Class']==0]['log_amount'].hist(bins=60, ax=axes[0,1], alpha=0.6, color='steelblue', density=True)
df_plot[df_plot['Class']==1]['log_amount'].hist(bins=60, ax=axes[0,1], alpha=0.8, color='crimson', density=True)
axes[0,1].set_title('log1p(Amount) Distribution')
axes[0,1].legend(['Legitimate', 'Fraud'])

# Boxplot by class
df.boxplot(column='Amount', by='Class', ax=axes[1,0], showfliers=True)
axes[1,0].set_title('Amount Boxplot by Class (raw)')
axes[1,0].set_xlabel('Class (0=Legit, 1=Fraud)')

# Outliers
p99 = df['Amount'].quantile(0.99)
p999 = df['Amount'].quantile(0.999)
print(f'\nAmount percentiles:')
for p in [50, 75, 90, 95, 99, 99.9, 100]:
    print(f'  p{p:5.1f}: ${df["Amount"].quantile(p/100):,.2f}')

# Fraud rate by amount bucket
df['amount_bucket'] = pd.cut(df['Amount'], bins=[0,10,50,100,500,1000,df['Amount'].max()], 
                              labels=['$0-10','$10-50','$50-100','$100-500','$500-1k','$1k+'])
fraud_by_amt = df.groupby('amount_bucket')['Class'].mean()
fraud_by_amt.plot(kind='bar', ax=axes[1,1], color='crimson', edgecolor='white')
axes[1,1].set_title('Fraud Rate by Amount Bucket')
axes[1,1].set_xlabel('Amount Range')
axes[1,1].set_ylabel('Fraud rate')
axes[1,1].tick_params(axis='x', rotation=30)

plt.suptitle('Amount Feature Analysis — RobustScaler vs StandardScaler vs Log?', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. V1–V28 Feature Analysis
Pipeline assumption: range ~[-30,30], no nulls, meaningful for model.

In [ ]:
v_cols = [f'V{i}' for i in range(1, 29)]

print('=== V FEATURE RANGES (actual vs pipeline assumption [-30,30]) ===')
v_stats = df[v_cols].agg(['min','max','mean','std']).T
v_stats['abs_max'] = v_stats[['min','max']].abs().max(axis=1)
print(v_stats.round(3).to_string())
print(f'\nGlobal min: {df[v_cols].min().min():.3f}')
print(f'Global max: {df[v_cols].max().max():.3f}')

In [ ]:
# Correlation with Class — which V features matter most?
correlations = df[v_cols + ['Amount','Time']].corrwith(df['Class']).abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

correlations.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('|Correlation| with Class (fraud)')
axes[0].set_ylabel('Absolute correlation')
axes[0].tick_params(axis='x', rotation=45)
axes[0].axhline(0.1, color='red', linestyle='--', alpha=0.5, label='0.1 threshold')
axes[0].legend()

# Heatmap of top correlated features
top_feats = correlations.head(10).index.tolist() + ['Class']
corr_matrix = df[top_feats].corr()
mask = np.triu(np.ones_like(corr_matrix), k=1)
sns.heatmap(corr_matrix, ax=axes[1], annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, mask=mask)
axes[1].set_title('Correlation Matrix — Top 10 Features + Class')

plt.suptitle('Feature Correlations with Fraud Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 10 features by |correlation| with Class:')
print(correlations.head(10).to_string())

In [ ]:
# Distribution shift: how different are fraud vs legit for top features?
top8 = correlations.head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(top8):
    legit = df[df['Class']==0][col]
    fraud = df[df['Class']==1][col]
    
    p1, p99 = df[col].quantile(0.01), df[col].quantile(0.99)
    bins = np.linspace(p1, p99, 50)
    
    axes[i].hist(legit.clip(p1,p99), bins=bins, alpha=0.5, density=True, color='steelblue', label='Legit')
    axes[i].hist(fraud.clip(p1,p99), bins=bins, alpha=0.7, density=True, color='crimson', label='Fraud')
    axes[i].set_title(f'{col}  (|r|={correlations[col]:.3f})')
    axes[i].legend(fontsize=7)
    axes[i].set_xlabel('')

plt.suptitle('Top 8 Features: Fraud vs Legitimate Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Validate Pipeline Preprocessing Assumptions

In [ ]:
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import train_test_split

print('=== STRATIFIED SPLIT CHECK ===')
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['Class'])
val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['Class'])

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    rate = split['Class'].mean()
    n_fraud = split['Class'].sum()
    print(f'  {name}: {len(split):>7,} rows | fraud={n_fraud:>4} ({rate:.4%})')

print(f'\nOverall fraud rate: {df["Class"].mean():.4%}')
print('✅ Stratification preserves fraud rate across splits')

print()
print('=== ROBUSTSCALER VS STANDARDSCALER ON AMOUNT ===')
amount = df[['Amount']]
rb = RobustScaler().fit_transform(amount)
st = StandardScaler().fit_transform(amount)
print(f'Amount raw:          min={df["Amount"].min():.2f}  max={df["Amount"].max():.2f}  std={df["Amount"].std():.2f}')
print(f'RobustScaler:        min={rb.min():.3f}  max={rb.max():.3f}  std={rb.std():.3f}')
print(f'StandardScaler:      min={st.min():.3f}  max={st.max():.3f}  std={st.std():.3f}')
print(f'\nAmount outlier rate (>10x IQR): {((df["Amount"] > df["Amount"].quantile(0.75)*10).sum()/len(df)):.4%}')
print('→ RobustScaler is correct choice if heavy outliers exist (verify above)')

## 6. Quick Model Benchmark
Validates AUPRC target of 0.85–0.90 and confirms XGBoost choice.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from xgboost import XGBClassifier

# Prep features (drop Time per pipeline assumption)
features = [c for c in df.columns if c not in ['Class', 'Time', 'time_hour', 'amount_bucket']]
X = df[features].copy()
X['Amount'] = RobustScaler().fit_transform(X[['Amount']])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

n_neg = (y_train==0).sum()
n_pos = (y_train==1).sum()
spw   = n_neg / n_pos
print(f'Train: {len(y_train):,} | Fraud: {n_pos} | scale_pos_weight: {spw:.1f}')

results = {}

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                                         scale_pos_weight=spw, eval_metric='aucpr',
                                         use_label_encoder=False, random_state=42, verbosity=0),
}

for name, model in models.items():
    print(f'\nTraining {name}...')
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)
    results[name] = {
        'AUPRC':   average_precision_score(y_test, probs),
        'ROC_AUC': roc_auc_score(y_test, probs),
        'F1':      f1_score(y_test, preds, zero_division=0),
    }
    print(f'  AUPRC={results[name]["AUPRC"]:.4f}  ROC-AUC={results[name]["ROC_AUC"]:.4f}  F1={results[name]["F1"]:.4f}')

print('\nPipeline assumption: XGBoost AUPRC in range [0.85, 0.90]')

In [ ]:
# Compare models visually
from sklearn.metrics import precision_recall_curve, roc_curve

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ['steelblue', 'darkorange', 'crimson']
for (name, model), color in zip(models.items(), colors):
    probs = model.predict_proba(X_test)[:, 1]
    
    # PR curve
    p, r, _ = precision_recall_curve(y_test, probs)
    axes[0].plot(r, p, color=color, linewidth=2,
                 label=f'{name} (AUPRC={results[name]["AUPRC"]:.3f})')
    
    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, probs)
    axes[1].plot(fpr, tpr, color=color, linewidth=2,
                 label=f'{name} (AUC={results[name]["ROC_AUC"]:.3f})')

axes[0].set_title('Precision-Recall Curve')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].legend(fontsize=8); axes[0].axhline(y=df['Class'].mean(), color='gray', linestyle='--', label='Baseline')

axes[1].plot([0,1],[0,1],'k--', alpha=0.3)
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(fontsize=8)

# Model comparison bar
metrics_df = pd.DataFrame(results).T
metrics_df.plot(kind='bar', ax=axes[2], color=['steelblue','darkorange','crimson'], edgecolor='white')
axes[2].set_title('Model Comparison')
axes[2].set_xticklabels(metrics_df.index, rotation=20, ha='right')
axes[2].legend(loc='lower right')
axes[2].axhline(0.85, color='red', linestyle='--', alpha=0.5, label='AUPRC target')

plt.suptitle('Model Benchmark — Validates Pipeline Choice', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Threshold Analysis
Validates pipeline assumption that 0.5 is wrong and cost-based sweep is better.

In [ ]:
from sklearn.metrics import precision_recall_curve, confusion_matrix

xgb = models['XGBoost']
probs = xgb.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, probs)

FN_COST = 10.0   # pipeline default
FP_COST = 1.0

n_pos = y_test.sum()
costs, f1s = [], []
for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds):
    if p == 0: continue
    tp = r * n_pos
    fn = n_pos - tp
    fp = tp/p - tp
    costs.append((t, FN_COST*fn + FP_COST*fp))
    preds_t = (probs >= t).astype(int)
    f1s.append((t, f1_score(y_test, preds_t, zero_division=0)))

best_t_cost = min(costs, key=lambda x: x[1])[0]
best_t_f1   = max(f1s, key=lambda x: x[1])[0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Cost curve
ts_c, cs = zip(*costs)
axes[0].plot(ts_c, cs, color='crimson', linewidth=2)
axes[0].axvline(best_t_cost, color='black', linestyle='--', label=f'Optimal t={best_t_cost:.3f}')
axes[0].axvline(0.5, color='gray', linestyle=':', label='t=0.5 (naive)')
axes[0].set_title(f'Business Cost vs Threshold\n(FN_cost={FN_COST}x FP_cost={FP_COST}x)')
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Total Cost')
axes[0].legend()

# F1 curve
ts_f, fs = zip(*f1s)
axes[1].plot(ts_f, fs, color='steelblue', linewidth=2)
axes[1].axvline(best_t_f1, color='black', linestyle='--', label=f'Best F1 t={best_t_f1:.3f}')
axes[1].axvline(0.5, color='gray', linestyle=':', label='t=0.5 (naive)')
axes[1].set_title('F1 Score vs Threshold')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('F1')
axes[1].legend()

# Confusion matrices side by side
for ax, t, label in [(None, 0.5, '0.5'), (None, best_t_cost, f'cost-optimal={best_t_cost:.3f}')]:
    pass

cm_naive = confusion_matrix(y_test, (probs>=0.5).astype(int))
cm_opt   = confusion_matrix(y_test, (probs>=best_t_cost).astype(int))
diff = cm_opt - cm_naive
sns.heatmap(diff, annot=True, fmt='+d', cmap='RdYlGn', ax=axes[2],
            xticklabels=['Pred Legit','Pred Fraud'], yticklabels=['True Legit','True Fraud'])
axes[2].set_title(f'CM Change: naive(t=0.5) → optimal(t={best_t_cost:.3f})\n(green=improvement)')

plt.suptitle('Threshold Optimisation — Cost-Based vs Naive 0.5', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Naive (t=0.5):          FN={cm_naive[1,0]}  FP={cm_naive[0,1]}  cost={FN_COST*cm_naive[1,0]+FP_COST*cm_naive[0,1]:.0f}')
print(f'Cost-optimal (t={best_t_cost:.3f}): FN={cm_opt[1,0]}  FP={cm_opt[0,1]}  cost={FN_COST*cm_opt[1,0]+FP_COST*cm_opt[0,1]:.0f}')

## 8. SHAP Feature Importance
Validates which features actually drive fraud predictions.

In [ ]:
import shap

xgb = models['XGBoost']
sample_idx = np.where(y_test == 1)[0][:200].tolist() + np.where(y_test == 0)[0][:800].tolist()
X_sample = X_test.iloc[sample_idx]

explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_sample)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Mean absolute SHAP
mean_abs = pd.Series(np.abs(shap_values).mean(axis=0), index=X_test.columns)
mean_abs.sort_values().tail(15).plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Mean |SHAP| — Top 15 Features')
axes[0].set_xlabel('Mean |SHAP value|')

# Summary beeswarm
plt.sca(axes[1])
shap.summary_plot(shap_values, X_sample, plot_type='dot', max_display=15,
                  show=False, color_bar=True)
axes[1].set_title('SHAP Summary — Feature Impact on Fraud Prediction')

plt.suptitle('SHAP Explainability — What Drives Fraud Predictions?', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 10 features by SHAP importance:')
print(mean_abs.sort_values(ascending=False).head(10).to_string())

## 9. Final Pipeline Assumption Audit

In [ ]:
print('=' * 65)
print('PIPELINE ASSUMPTION AUDIT')
print('=' * 65)

actual_fraud_rate = df['Class'].mean()
actual_spw = (df['Class']==0).sum() / (df['Class']==1).sum()
actual_v_min = df[v_cols].min().min()
actual_v_max = df[v_cols].max().max()
actual_rows = len(df)
actual_nulls = df.isnull().sum().sum()
actual_dupes = df.duplicated().sum()
actual_amount_outliers = (df['Amount'] > df['Amount'].quantile(0.75) * 10).sum()
actual_time_corr = abs(df['Time'].corr(df['Class']))

xgb_auprc = results['XGBoost']['AUPRC']

checks = [
    ('Fraud rate ~0.17%',         f'{actual_fraud_rate:.4%}',              abs(actual_fraud_rate - 0.0017) < 0.001),
    ('scale_pos_weight ~578',     f'{actual_spw:.1f}',                     abs(actual_spw - 578) < 50),
    ('V feature range [-30,30]',  f'[{actual_v_min:.1f}, {actual_v_max:.1f}]', actual_v_min > -50 and actual_v_max < 50),
    ('No nulls in features',      f'{actual_nulls} nulls',                 actual_nulls == 0),
    ('Row count ~284k',           f'{actual_rows:,}',                      actual_rows > 200_000),
    ('Amount needs RobustScaler', f'{actual_amount_outliers:,} outliers',  actual_amount_outliers > 100),
    ('Time has low signal',       f'|corr|={actual_time_corr:.4f}',        actual_time_corr < 0.05),
    ('XGBoost AUPRC 0.85-0.90',   f'{xgb_auprc:.4f}',                     0.75 <= xgb_auprc <= 0.95),
]

print(f'{'Assumption':<35} {'Actual':<25} Status')
print('-' * 65)
for assumption, actual, ok in checks:
    icon = '✅' if ok else '❌ REVISE'
    print(f'{assumption:<35} {actual:<25} {icon}')

print()
print('Any ❌ above = update the corresponding pipeline code')
print('Paste this output back to Claude for targeted fixes.')